In [1]:
import os
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
# --- Configuration ---
# Update these paths to match your local folders
CSV_PATH = r"C:\Users\My Computer\Desktop\sentiment analysis\TRAIN.csv"
AUDIO_DIR = r"C:\Users\My Computer\Desktop\sentiment analysis\TRAIN"

In [3]:
# Audio Hyperparameters
SAMPLE_RATE = 22050
DURATION = 3
N_MELS = 64
NUM_SAMPLES = SAMPLE_RATE * DURATION
HOP_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 0.001

In [4]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using Device: {device}")

🚀 Using Device: cuda


In [5]:
# --- 1. Custom Dataset Class ---
class AudioDataset(Dataset):
    def __init__(self, csv_file, audio_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.audio_dir = audio_dir
        
        # Verify columns
        if 'Class' not in self.df.columns and 'class' in self.df.columns:
            self.df.rename(columns={'class': 'Class'}, inplace=True)
            
        # Encode Labels
        self.le = LabelEncoder()
        self.df['label_encoded'] = self.le.fit_transform(self.df['Class'])
        
        # Save Label Encoder for later inference
        with open('label_encoder.pkl', 'wb') as f:
            pickle.dump(self.le, f)
        print(f"✅ Classes: {self.le.classes_}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get file path and label
        filename = str(self.df.iloc[idx]['Filename'])
        label = self.df.iloc[idx]['label_encoded']
        file_path = os.path.join(self.audio_dir, filename)

        # Load and process audio
        try:
            signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
            
            # Pad or Truncate
            if len(signal) > NUM_SAMPLES:
                signal = signal[:int(NUM_SAMPLES)]
            else:
                padding = int(NUM_SAMPLES) - len(signal)
                signal = np.pad(signal, (0, padding), mode='constant')

            # Extract Mel Spectrogram
            mel_spec = librosa.feature.melspectrogram(
                y=signal, sr=SAMPLE_RATE, n_mels=N_MELS, hop_length=HOP_LENGTH
            )
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
            
            # Normalize to 0-1
            norm_spec = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
            
            # Convert to PyTorch Tensor
            # Shape: (Channels, Freq, Time) -> (1, 64, 129)
            spec_tensor = torch.tensor(norm_spec, dtype=torch.float32).unsqueeze(0)
            
            return spec_tensor, torch.tensor(label, dtype=torch.long)
            
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            # Return a dummy zero tensor in case of error to prevent crash
            return torch.zeros((1, N_MELS, 129)), torch.tensor(0, dtype=torch.long)

In [6]:
# --- 2. Define Model Architecture ---
class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super(AudioCNN, self).__init__()
        
        # Block 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(16)
        )
        
        # Block 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(32)
        )
        
        # Block 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(64)
        )
        
        # Block 4
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(128)
        )
        
        # Classifier
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

In [20]:
# --- 3. Training Loop ---
# Prepare Data
print("⏳ Preparing Dataset...")
full_dataset = AudioDataset(CSV_PATH, AUDIO_DIR)
train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

num_classes = len(full_dataset.le.classes_)

# Initialize Model
model = AudioCNN(num_classes).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("🏋️ Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    # Print stats for the epoch
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f} | Accuracy: {100 * correct / total:.2f}%")

⏳ Preparing Dataset...
✅ Classes: ['Negative' 'Neutral' 'Positive']
🏋️ Starting Training...
Epoch [1/10] Loss: 0.8038 | Accuracy: 72.00%
Epoch [2/10] Loss: 0.4294 | Accuracy: 82.40%
Epoch [3/10] Loss: 0.3386 | Accuracy: 84.00%
Epoch [4/10] Loss: 0.3624 | Accuracy: 84.00%
Epoch [5/10] Loss: 0.2606 | Accuracy: 90.40%
Epoch [6/10] Loss: 0.1483 | Accuracy: 96.00%
Epoch [7/10] Loss: 0.0723 | Accuracy: 98.80%
Epoch [8/10] Loss: 0.0732 | Accuracy: 98.00%
Epoch [9/10] Loss: 0.1011 | Accuracy: 96.80%
Epoch [10/10] Loss: 0.0590 | Accuracy: 98.00%


In [22]:
# --- 4. Save Model ---
print("💾 Saving PyTorch Model...")
torch.save(model.state_dict(), 'audio_sentiment_model.pth')
print(f"✅ Model saved to: {os.path.abspath('audio_sentiment_model.pth')}")

💾 Saving PyTorch Model...
✅ Model saved to: c:\Users\My Computer\Desktop\sentiment analysis\audio_sentiment_model.pth
